# SCAGE Latent Representation Extraction

本 notebook 放置原本 `Evaluate_score_counting.ipynb` 中 `Testing score plot` 之前、與 latent representation 相關的內容。

In [ ]:
import os
import sys
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml
from rdkit.Chem import AllChem
from torch.utils.data import Dataset, DataLoader

In [ ]:
def _resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path('/workspace')]
    for c in candidates:
        if (c / '_config.py').exists() and (c / 'models').exists() and (c / 'data_process').exists():
            return c.resolve()
    raise RuntimeError('Cannot locate project root.')


PROJECT_ROOT = _resolve_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from _config import get_downstream_task_names, model_is_dp, pdir
from data_process.compound_tools import CompoundKit, mol_to_data_pkl
from data_process.data_collator import collator_finetune_pkl
from data_process.function_group_constant import nfg
from models.scage import Scage
from utils.global_var_util import GlobalVar

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
def ensure_cuda() -> str:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is required for SCAGE latent extraction.')
    return 'cuda'


def load_task_config(task_name: str) -> Dict:
    cfg_path = Path(pdir) / 'config' / 'config_finetune.yaml'
    config = yaml.load(open(cfg_path, 'r'), Loader=yaml.FullLoader)
    config['task_name'] = task_name
    config = get_downstream_task_names(config)
    return config


def build_scage(mode: str, task_name: str, device: str):
    config = load_task_config(task_name)
    if mode == 'pretrain_bin':
        GlobalVar.pretrain_task = ['finger', 'sp', 'angle', 'fg']
    model = Scage(
        mode=mode,
        atom_names=CompoundKit.atom_vocab_dict.keys(),
        atom_embed_dim=config['model']['atom_embed_dim'],
        num_kernel=config['model']['num_kernel'],
        layer_num=config['model']['layer_num'],
        num_heads=config['model']['num_heads'],
        atom_FG_class=nfg() + 1,
        hidden_size=config['model']['hidden_size'],
        num_tasks=config['num_tasks'],
    ).to(device)
    return model, config

In [ ]:
class SmilesFeatureDataset(Dataset):
    def __init__(self, smiles_list: List[str], num_tasks: int):
        self.data = []
        self.failed: List[Tuple[int, str]] = []
        for idx, smi in enumerate(smiles_list):
            mol = AllChem.MolFromSmiles(smi)
            if mol is None:
                self.failed.append((idx, smi))
                continue
            item = mol_to_data_pkl(mol)
            if item is None:
                self.failed.append((idx, smi))
                continue
            item['smiles'] = smi
            item['sample_id'] = idx
            item['label'] = np.zeros((num_tasks,), dtype=np.float32)
            self.data.append(item)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def encode_tokens(model: Scage, batched_data: Dict[str, torch.Tensor]) -> torch.Tensor:
    x = model.atom_feature(batched_data)
    for layer in model.EncoderAtomList:
        x = layer(
            x=x,
            attn_mask=batched_data['atom_attention_mask'],
            dist=batched_data['pair_distances'],
            dist_bar=batched_data['atom_dist_bar'],
        )
    return x

In [ ]:
# 範例：執行 latent 抽取
device = ensure_cuda()
task = 'caco2'
csv_path = './data/mpp/custom/caco2.csv'
smiles_col = 'SMILES'
mpp_ckpt = './weights/mpp/caco2.pth'

df = pd.read_csv(csv_path)
smiles_list = df[smiles_col].astype(str).tolist()[:128]

model, cfg = build_scage(mode='finetune', task_name=task, device=device)
print('Model ready. 可在此擴充 checkpoint 載入與 graph_token/atom_token 保存邏輯。')